# ML-05 - Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it - engineered features, categorical handling, fills.*

In [1]:
import pandas as pd

csv_path = "D:/Flyrank/Flyrank-assignments/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)
feature_vector = df[df["impressions_90d"] >= 1000].copy()

feature_vector["search_volume_filled"] = feature_vector["search_volume"].fillna(feature_vector["search_volume"].median())
feature_vector["content_type_filled"] = feature_vector["content_type"].fillna("unknown")
feature_vector["main_intent_filled"] = feature_vector["main_intent"].fillna("unknown")

final_feature_columns = [
    "avg_position",
    "content_age_days",
    "search_volume_filled",
    "content_type_filled",
    "main_intent_filled",
]

preview_columns = ["content_id", "client_id", *final_feature_columns, "ctr"]
print(f"Rows in the visible-page slice: {len(feature_vector):,}")
print("Five feature frame:", final_feature_columns)
feature_vector[preview_columns].head()

Rows in the visible-page slice: 13,512
Five feature frame: ['avg_position', 'content_age_days', 'search_volume_filled', 'content_type_filled', 'main_intent_filled']
             content_id          client_id  ...  main_intent_filled   ctr
0  content_304f48230142  client_f369cb89fc  ...       transactional  0.76
1  content_a1fb4e703a9e  client_4e07408562  ...       informational  0.05
2  content_9aa793d4d895  client_7f2253d7e2  ...       informational  0.09
3  content_331d6c4de07b  client_19581e27de  ...          commercial  0.49
4  content_d99b7a2d90ca  client_3fdba35f04  ...       informational  0.13

[5 rows x 8 columns]


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
feature_notes = pd.DataFrame(
    [
        {
            "feature": "avg_position",
            "meaning": "Observed average Google position over the same snapshot.",
            "missing_handling": "No missing values in the >=1000 impression slice.",
            "categorical": "No",
            "available_when": "At scoring time, because position is already observed in the export.",
            "available_before_scoring_is_true": True,
            "why_keep": "Expected CTR depends on rank, so this is the strongest non-leaky context signal.",
        },
        {
            "feature": "content_age_days",
            "meaning": "Days since the page was created.",
            "missing_handling": "No missing values in this slice.",
            "categorical": "No",
            "available_when": "Before scoring, from page metadata.",
            "available_before_scoring_is_true": True,
            "why_keep": "Older pages are more likely to need a refresh than very new pages.",
        },
        {
            "feature": "search_volume_filled",
            "meaning": "Keyword demand for the target topic after filling blanks with the slice median of 10.",
            "missing_handling": "Median fill after checking that blanks are rare in the visible slice.",
            "categorical": "No",
            "available_when": "Before scoring, from keyword metadata when it exists.",
            "available_before_scoring_is_true": True,
            "why_keep": "It helps separate low CTR on an important topic from low CTR on a tiny topic.",
        },
        {
            "feature": "content_type_filled",
            "meaning": "Page family such as keyword article, feedly article, or comparison article.",
            "missing_handling": "Fill missing values with 'unknown' even though this slice has none.",
            "categorical": "Yes",
            "available_when": "Before scoring, from content metadata.",
            "available_before_scoring_is_true": True,
            "why_keep": "Different page types have different baseline CTR behavior.",
        },
        {
            "feature": "main_intent_filled",
            "meaning": "Primary search intent such as informational or transactional.",
            "missing_handling": "Fill missing values with 'unknown'.",
            "categorical": "Yes",
            "available_when": "Before scoring, from the keyword brief when present.",
            "available_before_scoring_is_true": True,
            "why_keep": "Intent changes how likely a user is to click even at the same position.",
        },
    ]
)

feature_notes

                feature  ...                                           why_keep
0          avg_position  ...  Expected CTR depends on rank, so this is the s...
1      content_age_days  ...  Older pages are more likely to need a refresh ...
2  search_volume_filled  ...  It helps separate low CTR on an important topi...
3   content_type_filled  ...  Different page types have different baseline C...
4    main_intent_filled  ...  Intent changes how likely a user is to click e...

[5 rows x 7 columns]


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
import numpy as np
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

feature_vector["label_ctr"] = feature_vector["ctr"]

honest_columns = final_feature_columns.copy()
leaky_columns = honest_columns + ["label_ctr"]


def grouped_spearman(columns):
    X = feature_vector[columns].copy()
    y = feature_vector["ctr"].copy()
    groups = feature_vector["client_id"].copy()
    numeric_columns = [column for column in columns if X[column].dtype != "object"]
    categorical_columns = [column for column in columns if X[column].dtype == "object"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_columns),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )

    model = Pipeline([("preprocessor", preprocessor), ("ridge", Ridge(alpha=1.0))])
    predictions = cross_val_predict(
        model,
        X,
        y,
        cv=GroupKFold(n_splits=5),
        groups=groups,
        n_jobs=1,
    )
    return float(spearmanr(predictions, y).statistic), float(np.mean(np.abs(predictions - y)))

honest_spearman, honest_mae = grouped_spearman(honest_columns)
leaky_spearman, leaky_mae = grouped_spearman(leaky_columns)

leak_audit = pd.DataFrame(
    [
        {
            "version": "honest five-feature frame",
            "features_used": ", ".join(honest_columns),
            "spearman": round(honest_spearman, 3),
            "mae": round(honest_mae, 3),
            "why": "This is the number I keep because every feature is available at scoring time.",
        },
        {
            "version": "deliberate leak: add ctr itself",
            "features_used": ", ".join(leaky_columns),
            "spearman": round(leaky_spearman, 3),
            "mae": round(leaky_mae, 3),
            "why": "The score jumps to perfection because the model is reading the answer.",
        },
    ]
)

print("Leakage test grouped by client:")
print(leak_audit.to_string(index=False))
print("\nLeak removed. Final kept features:")
print(honest_columns)

Leakage test grouped by client:
                        version                                                                                            features_used  spearman   mae                                                                           why
      honest five-feature frame            avg_position, content_age_days, search_volume_filled, content_type_filled, main_intent_filled     0.316 0.206 This is the number I keep because every feature is available at scoring time.
deliberate leak: add ctr itself avg_position, content_age_days, search_volume_filled, content_type_filled, main_intent_filled, label_ctr     1.000 0.000        The score jumps to perfection because the model is reading the answer.

Leak removed. Final kept features:
['avg_position', 'content_age_days', 'search_volume_filled', 'content_type_filled', 'main_intent_filled']


## 4. What I excluded and why

*The list of fields you refused to use - with one line of why each.*

In [4]:
excluded_fields = pd.DataFrame(
    [
        {"field": "ctr", "why_excluded": "This is the proxy label, so using it as a feature is direct leakage."},
        {"field": "clicks_90d", "why_excluded": "CTR is computed from clicks and impressions, so clicks_90d is label-derived."},
        {"field": "impressions_90d", "why_excluded": "This is the other raw ingredient of ctr, so it belongs outside the model inputs."},
        {"field": "trend_direction", "why_excluded": "It is a different label family and would mix lanes in a confusing way."},
        {"field": "trend_pct", "why_excluded": "It is derived from the trend label inputs and is not part of my CTR opportunity story."},
        {"field": "content_id", "why_excluded": "It is an identifier for output rows, not a generalizable signal."},
        {"field": "client_id", "why_excluded": "It is for grouped validation only; using it as a feature would let the model memorize clients."},
        {"field": "provider_used", "why_excluded": "It is a production-history flag that may not hold up as a stable signal for editorial opportunity."},
        {"field": "model_used", "why_excluded": "It is a generation-history detail, not a clean pre-score behavior signal."},
    ]
)

excluded_fields

             field                                       why_excluded
0              ctr  This is the proxy label, so using it as a feat...
1       clicks_90d  CTR is computed from clicks and impressions, s...
2  impressions_90d  This is the other raw ingredient of ctr, so it...
3  trend_direction  It is a different label family and would mix l...
4        trend_pct  It is derived from the trend label inputs and ...
5       content_id  It is an identifier for output rows, not a gen...
6        client_id  It is for grouped validation only; using it as...
7    provider_used  It is a production-history flag that may not h...
8       model_used  It is a generation-history detail, not a clean...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.